In [1]:
import xarray as xr

ds = xr.open_dataset("nino34_year_month.nc")

print(ds)

<xarray.Dataset> Size: 3kB
Dimensions:  (year: 46, month: 12)
Coordinates:
  * year     (year) int64 368B 1980 1981 1982 1983 1984 ... 2022 2023 2024 2025
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
Data variables:
    nino34   (year, month) float32 2kB ...


In [1]:
import xarray as xr
import pandas as pd
import numpy as np

# =========================================================
# 1. 读取数据
# =========================================================
ds = xr.open_dataset("nino34_year_month.nc")

# 提取 nino34
nino = ds["nino34"]

# =========================================================
# 2. 计算 DJF 冬季平均
#    定义：
#    冬季年份 = 12月所在年份
#
#    例如：
#    1981 冬季 = 1981年12月 + 1982年1月 + 1982年2月
# =========================================================

winter_years = []
winter_means = []

years = nino.year.values

# 循环到倒数第二年，因为需要下一年的1、2月
for y in years[:-1]:

    try:
        dec = nino.sel(year=y, month=12)
        jan = nino.sel(year=y + 1, month=1)
        feb = nino.sel(year=y + 1, month=2)

        djf_mean = xr.concat([dec, jan, feb], dim="month").mean()

        winter_years.append(y)
        winter_means.append(float(djf_mean.values))

    except KeyError:
        # 如果缺数据则跳过
        continue

# =========================================================
# 3. 构建 DataFrame
# =========================================================
winter_df = pd.DataFrame({
    "winter_year": winter_years,
    "DJF_nino34": winter_means
})

print("\n=== DJF winter Niño3.4 index ===")
print(winter_df)

# =========================================================
# 4. 定义时间段并计算平均
# =========================================================
periods = {
    "1981-1990": (1981, 1990),
    "1991-2000": (1991, 2000),
    "2001-2010": (2001, 2010),
    "2011-2020": (2011, 2020),
    "2020-2024": (2020, 2024),
}

results = []

for name, (start, end) in periods.items():

    subset = winter_df[
        (winter_df["winter_year"] >= start) &
        (winter_df["winter_year"] <= end)
    ]

    mean_value = subset["DJF_nino34"].mean()

    results.append({
        "Period": name,
        "Mean_DJF_Nino34": mean_value
    })

# =========================================================
# 5. 输出最终表格
# =========================================================
result_df = pd.DataFrame(results)

print("\n=== Period Mean DJF Niño3.4 ===")
print(result_df)

# # =========================================================
# # 6. 保存为 CSV（可选）
# # =========================================================
# result_df.to_csv("DJF_Nino34_period_means.csv", index=False)

# # 如果希望保存逐年DJF指数：
# winter_df.to_csv("DJF_Nino34_yearly.csv", index=False)


=== DJF winter Niño3.4 index ===
    winter_year  DJF_nino34
0          1980   -0.183333
1          1981   -0.033333
2          1982    2.326667
3          1983   -0.760000
4          1984   -1.110000
5          1985   -0.610000
6          1986    1.113333
7          1987    0.696667
8          1988   -1.766667
9          1989    0.083333
10         1990    0.383333
11         1991    1.616667
12         1992    0.246667
13         1993    0.023333
14         1994    1.060000
15         1995   -0.676667
16         1996   -0.366667
17         1997    2.266667
18         1998   -1.483333
19         1999   -1.620000
20         2000   -0.810000
21         2001   -0.200000
22         2002    1.010000
23         2003    0.250000
24         2004    0.510000
25         2005   -0.813333
26         2006    0.603333
27         2007   -1.700000
28         2008   -0.870000
29         2009    1.473333
30         2010   -1.530000
31         2011   -0.863333
32         2012   -0.316667
33         201